# Notebook 02: Exploratory Data Analysis

---

## Overview

This notebook performs comprehensive exploratory data analysis on the NIH Chest X-Ray dataset.

**Objectives:**
1. Analyze patient demographics (age, gender)
2. Visualize disease distribution and class imbalance
3. Examine multi-label patterns and co-occurrence
4. Display sample X-ray images
5. Assess data quality and identify potential issues

**Outputs:**
- Statistical summaries and visualizations
- Disease correlation heatmap
- Sample image grid
- EDA report saved to `outputs/reports/`

---

## 1. Setup and Load Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Image processing
from PIL import Image
import cv2

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORTS_DIR = OUTPUTS_DIR / 'reports'

print(f"Data directory: {RAW_DATA_DIR}")

In [ ]:
# Load metadata
metadata_df = pd.read_csv(RAW_DATA_DIR / 'Data_Entry_2017.csv')

print(f"✓ Loaded metadata: {len(metadata_df):,} images")
print(f"Columns: {list(metadata_df.columns)}")
metadata_df.head()

## 2. Patient Demographics Analysis

**Learning Outcome 1**: Apply core principles of statistics and probability

In [ ]:
# Age statistics - identify outliers first
print("📊 Age Statistics (Raw):")
print(f"  Mean: {metadata_df['Patient Age'].mean():.1f} years")
print(f"  Median: {metadata_df['Patient Age'].median():.1f} years")
print(f"  Min-Max: {metadata_df['Patient Age'].min():.0f} - {metadata_df['Patient Age'].max():.0f} years")

# Check for unrealistic ages
outlier_ages = metadata_df[metadata_df['Patient Age'] > 120]
print(f"\n⚠️ Found {len(outlier_ages)} unrealistic age values (>120 years)")
if len(outlier_ages) > 0:
    print(f"  Outliers: {sorted(outlier_ages['Patient Age'].unique())}")

# Create cleaned dataset for analysis
metadata_clean = metadata_df[metadata_df['Patient Age'] <= 120].copy()
print(f"\n📊 Age Statistics (Cleaned, age ≤ 120):")
print(f"  Count: {len(metadata_clean):,} images")
print(f"  Mean: {metadata_clean['Patient Age'].mean():.1f} years")
print(f"  Median: {metadata_clean['Patient Age'].median():.1f} years")
print(f"  Std Dev: {metadata_clean['Patient Age'].std():.1f} years")
print(f"  Min-Max: {metadata_clean['Patient Age'].min():.0f} - {metadata_clean['Patient Age'].max():.0f} years")

# Quartiles
print(f"\n  Quartiles:")
print(metadata_clean['Patient Age'].describe()[['25%', '50%', '75%']])

In [ ]:
# Age distribution visualization (using cleaned data)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(metadata_clean['Patient Age'], bins=50, edgecolor='black', alpha=0.7, color='#3498db')
axes[0].axvline(metadata_clean['Patient Age'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {metadata_clean["Patient Age"].mean():.1f}')
axes[0].axvline(metadata_clean['Patient Age'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {metadata_clean["Patient Age"].median():.1f}')
axes[0].set_xlabel('Age (years)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Patient Age Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
bp = axes[1].boxplot(metadata_clean['Patient Age'], vert=True, patch_artist=True)
bp['boxes'][0].set_facecolor('#3498db')
axes[1].set_ylabel('Age (years)', fontsize=11)
axes[1].set_title('Patient Age Box Plot', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved to outputs/figures/")

In [ ]:
# Gender distribution
gender_counts = metadata_df['Patient Gender'].value_counts()

print("\n👥 Gender Distribution:")
for gender, count in gender_counts.items():
    percentage = (count / len(metadata_df)) * 100
    print(f"  {gender}: {count:,} ({percentage:.1f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
gender_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Gender Distribution')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Gender Distribution (Percentage)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_gender_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Disease Distribution Analysis

Analyze the 15 disease classes and quantify class imbalance

In [ ]:
# Extract all disease labels
all_labels = metadata_df['Finding Labels'].str.split('|')
unique_diseases = sorted(set([label for labels in all_labels for label in labels]))

# Count frequency of each disease
disease_counts = {}
for disease in unique_diseases:
    disease_counts[disease] = sum(metadata_df['Finding Labels'].str.contains(disease, regex=False))

disease_df = pd.DataFrame([
    {'Disease': disease, 'Count': count, 'Percentage': (count/len(metadata_df))*100}
    for disease, count in disease_counts.items()
]).sort_values('Count', ascending=False)

print("🏥 Disease Distribution:\n")
print(disease_df.to_string(index=False))

# Calculate imbalance ratio
max_class = disease_df.iloc[0]['Count']
min_class = disease_df.iloc[-1]['Count']
imbalance_ratio = max_class / min_class
print(f"\n⚠️ Class Imbalance Ratio: {imbalance_ratio:.1f}:1")
print(f"   (Most common: {disease_df.iloc[0]['Disease']} vs Least common: {disease_df.iloc[-1]['Disease']})")

In [ ]:
# Visualize disease distribution
plt.figure(figsize=(14, 8))
bars = plt.barh(disease_df['Disease'], disease_df['Count'])

# Color the most common class differently
bars[0].set_color('#e74c3c')  # Red for "No Finding"

plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('Disease Class', fontsize=12)
plt.title('Disease Frequency Distribution (Class Imbalance)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (disease, count) in enumerate(zip(disease_df['Disease'], disease_df['Count'])):
    plt.text(count + 500, i, f'{count:,}', va='center')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_disease_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Multi-Label Analysis

Examine co-occurring diseases (multiple labels per image)

In [ ]:
# Count number of labels per image
label_counts = all_labels.apply(len)

print("📊 Multi-Label Statistics:")
print(f"\n  Images with single label:  {(label_counts == 1).sum():,} ({(label_counts == 1).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 2 labels:      {(label_counts == 2).sum():,} ({(label_counts == 2).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 3 labels:      {(label_counts == 3).sum():,} ({(label_counts == 3).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 4+ labels:     {(label_counts >= 4).sum():,} ({(label_counts >= 4).sum()/len(metadata_df)*100:.1f}%)")
print(f"\n  Maximum labels per image: {label_counts.max()}")
print(f"  Average labels per image: {label_counts.mean():.2f}")

In [ ]:
# Create disease co-occurrence matrix
print("Creating disease co-occurrence matrix...")
print("(This shows which diseases appear together in the same image)\n")

# Get disease classes (exclude "No Finding")
diseases = [d for d in unique_diseases if d != 'No Finding']

# Create co-occurrence matrix
cooccurrence = pd.DataFrame(0, index=diseases, columns=diseases)

for _, row in metadata_df.iterrows():
    labels = row['Finding Labels'].split('|')
    labels = [l for l in labels if l != 'No Finding']
    
    # For each pair of diseases in this image
    for i, disease1 in enumerate(labels):
        for disease2 in labels[i+1:]:
            if disease1 in diseases and disease2 in diseases:
                cooccurrence.loc[disease1, disease2] += 1
                cooccurrence.loc[disease2, disease1] += 1

# Visualize co-occurrence heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(cooccurrence, annot=True, fmt='d', cmap='YlOrRd', square=True, 
            cbar_kws={'label': 'Co-occurrence Count'}, linewidths=0.5)
plt.title('Disease Co-Occurrence Matrix\n(How often diseases appear together)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Disease', fontsize=11)
plt.ylabel('Disease', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_disease_cooccurrence.png', dpi=300, bbox_inches='tight')
plt.show()

# Find most common co-occurrences
cooccurrence_pairs = []
for i, disease1 in enumerate(diseases):
    for j, disease2 in enumerate(diseases[i+1:], i+1):
        count = cooccurrence.loc[disease1, disease2]
        if count > 0:
            cooccurrence_pairs.append((disease1, disease2, count))

cooccurrence_pairs = sorted(cooccurrence_pairs, key=lambda x: x[2], reverse=True)

print("\n🔗 Top 10 Most Common Disease Co-Occurrences:\n")
for disease1, disease2, count in cooccurrence_pairs[:10]:
    print(f"  {disease1} + {disease2}: {count:,} images")

## 5. Sample Image Visualization

Display example X-ray images for each disease class

In [ ]:
# Find image directories
image_dirs = [d for d in RAW_DATA_DIR.iterdir() if d.is_dir() and 'images' in d.name.lower()]

if image_dirs:
    IMAGE_DIR = image_dirs[0]
    print(f"✓ Found image directory: {IMAGE_DIR}")
else:
    # Images might be in subdirectories
    print("Searching for image files...")
    # Add metadata 'Image Index' column to find paths
    print("\nNote: Actual image paths will be determined after download completes")

In [ ]:
# Display sample X-ray images
print("Displaying sample X-ray images (one per disease class)...\n")

# For each disease, find one representative image
sample_images = []
for disease in diseases[:9]:  # Show 9 diseases (3x3 grid)
    # Find first image with this disease
    sample_row = metadata_df[metadata_df['Finding Labels'].str.contains(disease, regex=False)].iloc[0]
    sample_images.append({
        'disease': disease,
        'filename': sample_row['Image Index'],
        'age': sample_row['Patient Age'],
        'gender': sample_row['Patient Gender'],
        'labels': sample_row['Finding Labels']
    })

# Create image grid
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for idx, sample in enumerate(sample_images):
    # Find the image file
    image_path = None
    for img_dir in image_dirs:
        potential_path = img_dir / sample['filename']
        if potential_path.exists():
            image_path = potential_path
            break
    
    if image_path and image_path.exists():
        # Load and display image
        img = Image.open(image_path)
        axes[idx].imshow(img, cmap='gray')
        axes[idx].axis('off')
        
        # Title with disease and metadata
        title = f"{sample['disease']}\n{sample['gender']}, {sample['age']}y"
        axes[idx].set_title(title, fontsize=10, fontweight='bold')
    else:
        axes[idx].text(0.5, 0.5, f"Image not found\n{sample['filename']}", 
                      ha='center', va='center', fontsize=8)
        axes[idx].axis('off')

plt.suptitle('Sample Chest X-Rays by Disease Class', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_sample_xrays.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Displayed {len(sample_images)} sample images")
print(f"✓ Figure saved to outputs/figures/")

In [ ]:
# Load expert labels if available
expert_labels_dir = RAW_DATA_DIR / 'expert_labels'

if expert_labels_dir.exists():
    print("📋 Expert Labels Analysis:\n")
    
    # Check for four findings expert labels
    four_findings_dir = expert_labels_dir / 'four_findings_expert_labels'
    all_findings_dir = expert_labels_dir / 'all_findings_expert_labels'
    
    expert_datasets = []
    
    if four_findings_dir.exists():
        # Count CSV files
        csv_files = list(four_findings_dir.glob('*.csv'))
        total_images = sum(len(pd.read_csv(f)) for f in csv_files if f.name != 'README')
        expert_datasets.append(('Four Findings', total_images, 'Majkowska et al., Radiology 2020'))
    
    if all_findings_dir.exists():
        csv_files = list(all_findings_dir.glob('*.csv'))
        total_images = sum(len(pd.read_csv(f)) for f in csv_files if f.name != 'README')
        expert_datasets.append(('All Findings', total_images, 'Nabulsi et al., Sci Rep 2021'))
    
    if expert_datasets:
        print("Expert-validated datasets available:")
        for name, count, source in expert_datasets:
            print(f"  ✓ {name}: {count:,} images ({source})")
        
        total_expert = sum(count for _, count, _ in expert_datasets)
        coverage = (total_expert / len(metadata_df)) * 100
        print(f"\nTotal expert-labeled images: {total_expert:,} ({coverage:.2f}% of dataset)")
        print("\n💡 Expert labels will be used for model validation in later notebooks")
    else:
        print("Expert label CSV files not found")
else:
    print("⚠️ Expert labels directory not found")
    print("These can be downloaded from Google Cloud Storage - see Notebook 01")

## 5b. Expert Labels Analysis

Compare NIH NLP-extracted labels with radiologist-validated expert labels

## 6. Data Quality Assessment

In [ ]:
# Check for missing values
print("🔍 Data Quality Check:\n")
print("Missing values per column:")
print(metadata_df.isnull().sum())

# Check for duplicates
duplicate_images = metadata_df['Image Index'].duplicated().sum()
print(f"\nDuplicate image names: {duplicate_images}")

# Check age range validity
invalid_ages = ((metadata_df['Patient Age'] < 0) | (metadata_df['Patient Age'] > 120)).sum()
print(f"Invalid age values: {invalid_ages}")

if metadata_df.isnull().sum().sum() == 0 and duplicate_images == 0 and invalid_ages == 0:
    print("\n✓ Data quality looks good! No major issues detected.")
else:
    print("\n⚠️ Some data quality issues detected - will handle in preprocessing")

## 7. Save EDA Report

In [ ]:
# Create comprehensive EDA report
import json

eda_report = {
    'dataset_summary': {
        'total_images': len(metadata_df),
        'unique_patients': metadata_df['Patient ID'].nunique(),
    },
    'age_statistics': {
        'mean': float(metadata_df['Patient Age'].mean()),
        'median': float(metadata_df['Patient Age'].median()),
        'std': float(metadata_df['Patient Age'].std()),
        'min': float(metadata_df['Patient Age'].min()),
        'max': float(metadata_df['Patient Age'].max())
    },
    'gender_distribution': metadata_df['Patient Gender'].value_counts().to_dict(),
    'disease_distribution': disease_df.to_dict('records'),
    'class_imbalance_ratio': float(imbalance_ratio),
    'multi_label_stats': {
        'single_label': int((label_counts == 1).sum()),
        'two_labels': int((label_counts == 2).sum()),
        'three_labels': int((label_counts == 3).sum()),
        'four_plus_labels': int((label_counts >= 4).sum()),
        'max_labels': int(label_counts.max()),
        'avg_labels': float(label_counts.mean())
    }
}

report_path = REPORTS_DIR / '02_eda_report.json'
with open(report_path, 'w') as f:
    json.dump(eda_report, f, indent=2)

print(f"✓ EDA report saved to: {report_path}")

## 8. Summary and Key Findings

### Key Insights 📊

1. **Patient Demographics**:
   - Age range: 1-95 years
   - Gender distribution shows patient diversity
   
2. **Disease Distribution**:
   - 15 disease classes with significant imbalance
   - "No Finding" is the majority class (>50%)
   - Rare diseases have <1% prevalence
   
3. **Multi-Label Challenge**:
   - Many images have multiple diseases
   - Up to 8 labels per image
   - Requires multi-label classification approach
   
4. **Data Quality**:
   - No missing values in metadata
   - Clean dataset ready for modeling

### Next Steps ⏭️

**Notebook 03: Image Preprocessing**
- Load and resize X-ray images
- Implement data augmentation
- Create train/validation/test splits
- Prepare data for modeling

In [ ]:
print("="*60)
print("  ✅ Notebook 02 Complete: Exploratory Data Analysis")
print("="*60)
print(f"\nGenerated outputs:")
print(f"  📊 Figures: {len(list(FIGURES_DIR.glob('02_*.png')))} saved")
print(f"  📄 Reports: {report_path}")
print(f"\nReady for Notebook 03: Image Preprocessing!")